# M6 · Support Tickets Cleaning, Integrity Checks and Clean Database (SQL, PostgreSQL)

**Goal:** clean the 30,000 support tickets **in SQL**, check that all five clean tables fit together, and deliver the complete clean data as a PostgreSQL database.

The logic lives in plain `.sql` files next to this notebook, so it can be read, reviewed and run in any SQL client (DBeaver, psql):

| File | What it does |
|---|---|
| `01_create_tables.sql` | schemas `raw` (as delivered), `work` (intermediate), `clean` (deliverable) and all table definitions |
| `02_clean_tickets.sql` | CTE pipeline: `TRIM`, `NULLIF`, `ROW_NUMBER`, `CASE`/`COALESCE` mapping (incl. `REVERSE` for reversed text), safe `CAST` with `pg_input_is_valid`, rebuilding a missing timestamp from `resolution_time_hours`, lookup-based typo fix for agent names |
| `03_ticket_flags.sql` | one row per ticket problem, typed as needs review / invalid or missing value / auto-fixed (info) |
| `04_publish_tickets.sql` | final `clean.tickets` table (clean columns + `needs_review` + `dq_flags`) |
| `05_integrity_checks.sql` | 15 checks across all tables: primary keys, foreign keys (orphans), business rules |

This notebook loads the data, runs the files in order and exports the results.
The connection password is read by PostgreSQL's client from `pgpass.conf`; it is never written in code.

**Outputs**
| File | Folder | Content |
|---|---|---|
| `tickets_clean.xlsx` | `clean_data/` | sheets `tickets` and `flagged` |
| `tickets_clean.csv` | `clean_data/csv/` | the same clean tickets as UTF-8 CSV |
| `customer360_clean.dump` | `clean_data/database/` | schema `clean` with all 6 clean tables (restore with `pg_restore`) |
| `m6_fix_log.xlsx` | `output/` (this folder) | fix log, integrity checks, validation |

## 1. Setup and database

In [1]:
import shutil
import subprocess
from pathlib import Path

import pandas as pd
import psycopg

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 200)
pd.set_option("display.max_colwidth", 80)

RAW_DIR = Path("../../01_data_and_task/raw_data")
CLEAN_DIR = Path("../../03_clean_data_and_report/clean_data")
OUT_DIR = Path("output")  # detailed working outputs of this milestone (logs)
OUT_DIR.mkdir(exist_ok=True)
ISSUE_LOG = Path("../01_profiling-pandas/output/issue_log.xlsx")
SQL_DIR = Path(".")
DB = "customer360"
CONN = "host=localhost port=5432 user=postgres"  # password comes from pgpass.conf

with psycopg.connect(f"{CONN} dbname=postgres", autocommit=True) as admin:
    exists = admin.execute("SELECT 1 FROM pg_database WHERE datname = %s", (DB,)).fetchone()
    if not exists:
        admin.execute(f"CREATE DATABASE {DB}")
    print(f"database {DB!r}:", "already existed" if exists else "created")

conn = psycopg.connect(f"{CONN} dbname={DB}", autocommit=True)


def run_sql(file):
    conn.execute((SQL_DIR / file).read_text(encoding="utf-8"))
    print("ran", file)


def query(sql):
    cur = conn.execute(sql)
    return pd.DataFrame(cur.fetchall(), columns=[c.name for c in cur.description])


issue_log = pd.read_excel(ISSUE_LOG, sheet_name="issue_log")
m6_issues = issue_log[issue_log["milestone"] == "M6"]
m6_issues[["issue_id", "column", "issue", "rows_affected"]]

database 'customer360': already existed


,issue_id,column,issue,rows_affected
69,ISS-070,sentiment,"non-standard value (15 variants: case, leetspeak, reversed, truncated, suffi...",9000
70,ISS-071,issue_type,"non-standard value (31 variants: case, leetspeak, reversed, truncated, suffi...",8739
71,ISS-072,ticket_created,"impossible or unparseable timestamp (3 patterns: month 15/31, hour 28)",5390
72,ISS-073,ticket_resolved,"impossible or unparseable timestamp (3 patterns: month 15/31, hour 28)",5324
73,ISS-074,resolution_time_hours,resolution_time_hours does not match resolved minus created (more than 1 hou...,2689
74,ISS-075,support_agent,leading/trailing spaces,2574
75,ISS-076,support_agent,noise characters in agent name (e.g. trailing '--'),2572
76,ISS-077,issue_type,leading/trailing spaces,2245
77,ISS-078,sentiment,leading/trailing spaces,2206
78,ISS-079,ticket_resolved,timestamp after the data export date (future),1975


## 2. Create tables and load data

- The raw tickets CSV is loaded with `COPY` into `raw.support_tickets` (all text, nothing converted).
- The complete clean tables from M2-M5 are loaded into `clean.*` (every column, every row). Values are read as text and PostgreSQL converts them to the column types, so a wrong value would stop the load instead of slipping in.

In [2]:
run_sql("01_create_tables.sql")

with conn.cursor() as cur:
    with cur.copy("COPY raw.support_tickets FROM STDIN WITH (FORMAT csv, HEADER true)") as copy:
        copy.write((RAW_DIR / "support_tickets_30000_dirty.csv").read_bytes())


def load(table, path, sheet, columns=None):
    """Bulk-load one Excel sheet with COPY; empty cells become NULL."""
    df = pd.read_excel(path, sheet_name=sheet, dtype=str, usecols=columns)
    with conn.cursor() as cur:
        with cur.copy(f"COPY {table} ({', '.join(df.columns)}) FROM STDIN WITH (FORMAT csv)") as copy:
            copy.write(df.to_csv(index=False, header=False))
    print(f"{table:<24} {len(df):>8,} rows x {df.shape[1]} columns")


load("clean.customers", CLEAN_DIR / "customers_clean.xlsx", "customers")
load("clean.customer_devices", CLEAN_DIR / "customers_clean.xlsx", "customer_devices")
load("clean.products", CLEAN_DIR / "products_clean.xlsx", "products_clean",
     ["product_id", "product_name", "category", "price", "price_status", "flag_type", "review_reason"])
load("clean.orders", CLEAN_DIR / "orders_clean.xlsx", "orders")
load("clean.clickstream", CLEAN_DIR / "clickstream_clean.xlsx", "clickstream")
query("SELECT COUNT(*) AS raw_tickets FROM raw.support_tickets")

ran 01_create_tables.sql


clean.customers            48,200 rows x 20 columns


clean.customer_devices     51,854 rows x 2 columns
clean.products                500 rows x 7 columns


clean.orders              300,000 rows x 11 columns


clean.clickstream         500,000 rows x 11 columns


,raw_tickets
0,30000


## 3. Clean the tickets (`02_clean_tickets.sql`)

Before the main result, the key assumption of step 5 is shown: for every ticket where both timestamps are valid, `resolved - created` equals `resolution_time_hours`. Because this holds for all of them, a missing timestamp can be rebuilt exactly from the other one.

In [3]:
run_sql("02_clean_tickets.sql")

query("""
SELECT COUNT(*) AS tickets_with_both_valid_timestamps,
       SUM(CASE WHEN ABS(EXTRACT(EPOCH FROM ticket_resolved - ticket_created) / 3600
                         - resolution_time_hours) < 0.01 THEN 1 ELSE 0 END) AS hours_match
FROM work.support_tickets
WHERE NOT created_is_derived AND NOT resolved_is_derived
  AND ticket_created IS NOT NULL AND ticket_resolved IS NOT NULL
""")

ran 02_clean_tickets.sql


,tickets_with_both_valid_timestamps,hours_match
0,14586,14586


In [4]:
query("""
SELECT issue_type_raw, issue_type, COUNT(*) AS tickets
FROM work.support_tickets
GROUP BY 1, 2
ORDER BY 2, 3 DESC
""")

,issue_type_raw,issue_type,tickets
0,delay,delay,5820
1,delayx,delay,294
2,delay_,delay,284
3,DELAY,delay,275
4,d3lay,delay,255
5,yaled,delay,253
6,del,delay,241
7,payment,payment,5747
8,tnemyap,payment,306
9,paymentx,payment,289


In [5]:
query("""
SELECT sentiment_raw, sentiment, COUNT(*) AS tickets
FROM work.support_tickets
GROUP BY 1, 2
ORDER BY 2, 3 DESC
""")

,sentiment_raw,sentiment,tickets
0,negative,negative,7771
1,neg,negative,1497
2,NEGATIVE,negative,405
3,n3gativ3,negative,346
4,neutral,neutral,7660
5,neu,neutral,1509
6,NEUTRAL,neutral,411
7,n3utral,neutral,387
8,positive,positive,7775
9,pos,positive,1474


In [6]:
query("""
SELECT agent_raw, support_agent, agent_typo_fixed
FROM work.support_tickets
WHERE agent_raw <> support_agent
ORDER BY agent_typo_fixed DESC, support_agent
LIMIT 15
""")

,agent_raw,support_agent,agent_typo_fixed
0,Aaaron Mcpherson,Aaron Mcpherson,True
1,Aaaron Mcpherson,Aaron Mcpherson,True
2,Aaaron Mcpherson,Aaron Mcpherson,True
3,Aaaron Mcpherson,Aaron Mcpherson,True
4,Aaaron Mcpherson,Aaron Mcpherson,True
5,Aaaron Mcpherson,Aaron Mcpherson,True
6,Aaaron Mcpherson,Aaron Mcpherson,True
7,Aaaron Mcpherson,Aaron Mcpherson,True
8,Aliciaa Simpson,Alicia Simpson,True
9,Aliciaa Simpson,Alicia Simpson,True


## 4. Flags and final table (`03_ticket_flags.sql`, `04_publish_tickets.sql`)

In [7]:
run_sql("03_ticket_flags.sql")
run_sql("04_publish_tickets.sql")
query("SELECT flag_type, reason, COUNT(*) AS tickets FROM clean.ticket_flags GROUP BY 1, 2 ORDER BY 1, 3 DESC")

ran 03_ticket_flags.sql


ran 04_publish_tickets.sql


,flag_type,reason,tickets
0,auto-fixed (info),rebuilt as ticket_resolved minus resolution_time_hours,6292
1,auto-fixed (info),rebuilt as ticket_created plus resolution_time_hours,6272
2,invalid or missing value,impossible timestamp (bad month/hour); set to null,10714
3,invalid or missing value,timestamp after the data export (2025-12-02); set to null,3796
4,needs review,"possible doubled-letter typo; no clean spelling of this agent exists, kept a...",139


## 5. Integrity checks across all tables (`05_integrity_checks.sql`)

`pass` = no failing rows. `flag` = rows that contradict another table; they are reported for the client to decide, not changed.

How to read the three flags:
- **Device check (100%)**: the device IDs in the clickstream never match the devices in the CRM, so web events can be linked to customers by `customer_id` only, not by device.
- **Orders / tickets before signup (about 8%)**: activity dated before the customer's signup date. Either the signup date or the activity date is wrong; the source cannot tell which, so both are kept.

In [8]:
run_sql("05_integrity_checks.sql")
integrity = query("SELECT * FROM clean.integrity_checks ORDER BY check_no")
integrity

ran 05_integrity_checks.sql


,check_no,check_type,check_name,failing_rows,total_rows,failing_pct,status
0,1,primary key,customers.customer_id is unique,0,48200,0.00,pass
1,2,primary key,products.product_id is unique,0,500,0.00,pass
2,3,primary key,orders.order_id is unique,0,300000,0.00,pass
3,4,primary key,tickets.ticket_id is unique,0,30000,0.00,pass
4,5,primary key,clickstream.event_id is unique,0,500000,0.00,pass
5,6,foreign key,orders.customer_id exists in customers,0,300000,0.00,pass
6,7,foreign key,orders.product_id exists in products,0,300000,0.00,pass
7,8,foreign key,tickets.customer_id exists in customers,0,30000,0.00,pass
8,9,foreign key,clickstream.customer_id exists in customers (anonymous events excluded),0,349447,0.00,pass
9,10,foreign key,clickstream.product_id exists in products,0,368942,0.00,pass


## 6. Fix log

In [9]:
s = query("""
SELECT
  SUM(CASE WHEN issue_type_raw <> issue_type THEN 1 ELSE 0 END)                          AS issue_mapped,
  SUM(CASE WHEN sentiment_raw <> sentiment THEN 1 ELSE 0 END)                            AS sentiment_mapped,
  SUM(CASE WHEN created_raw IS NOT NULL AND created_parsed IS NULL THEN 1 ELSE 0 END)    AS created_impossible,
  SUM(CASE WHEN resolved_raw IS NOT NULL AND resolved_parsed IS NULL THEN 1 ELSE 0 END)  AS resolved_impossible,
  SUM(CASE WHEN created_parsed >= '2025-12-03' THEN 1 ELSE 0 END)                        AS created_future,
  SUM(CASE WHEN resolved_parsed >= '2025-12-03' THEN 1 ELSE 0 END)                       AS resolved_future,
  SUM(CASE WHEN created_is_derived THEN 1 ELSE 0 END)                                    AS created_derived,
  SUM(CASE WHEN resolved_is_derived THEN 1 ELSE 0 END)                                   AS resolved_derived,
  SUM(CASE WHEN created_raw IS NULL AND created_is_derived THEN 1 ELSE 0 END)            AS created_blank_rebuilt,
  SUM(CASE WHEN resolved_raw IS NULL AND resolved_is_derived THEN 1 ELSE 0 END)          AS resolved_blank_rebuilt,
  SUM(CASE WHEN agent_typo_fixed THEN 1 ELSE 0 END)                                      AS agent_typo_fixed,
  SUM(CASE WHEN support_agent ~ 'aa' AND NOT agent_typo_fixed THEN 1 ELSE 0 END)         AS agent_typo_kept,
  SUM(CASE WHEN agent_raw ~ '[^A-Za-z .''-]|--|-$' THEN 1 ELSE 0 END)                    AS agent_noise,
  SUM(CASE WHEN ticket_resolved < ticket_created THEN 1 ELSE 0 END)                      AS resolved_before_created_left
FROM work.support_tickets
""").iloc[0].astype(int)

raw_t = pd.read_csv(RAW_DIR / "support_tickets_30000_dirty.csv", dtype=str, keep_default_na=False)
spaces = {c: int(raw_t[c].ne(raw_t[c].str.strip()).sum()) for c in raw_t.columns}
REBUILT = "; rebuilt from resolution_time_hours when the other timestamp is valid (see derived flags)"

plan = [
    ("sentiment", "non-standard value", s.sentiment_mapped, 0,
     "Mapped with CASE/COALESCE: exact, leetspeak 3->e, unique prefix (neu/neg/pos)"),
    ("issue_type", "non-standard value", s.issue_mapped, 0,
     "Mapped with CASE/COALESCE: exact, REVERSE() for reversed text, leetspeak, suffix x/_, unique prefix"),
    ("sentiment", "leading/trailing spaces", spaces["sentiment"], 0, "TRIM"),
    ("issue_type", "leading/trailing spaces", spaces["issue_type"], 0, "TRIM"),
    ("support_agent", "leading/trailing spaces", spaces["support_agent"], 0, "TRIM"),
    ("support_agent", "noise characters", s.agent_noise, 0, "REGEXP_REPLACE removed symbols ('--', '@@')"),
    ("support_agent", "possible doubled-letter typo", s.agent_typo_fixed, s.agent_typo_kept,
     "Fixed only where the 'a' spelling exists as another agent name; others kept and flagged"),
    ("ticket_created", "impossible or unparseable", 0, s.created_impossible,
     "Not cast (pg_input_is_valid false): null, flagged" + REBUILT),
    ("ticket_resolved", "impossible or unparseable", 0, s.resolved_impossible,
     "Not cast (pg_input_is_valid false): null, flagged" + REBUILT),
    ("ticket_created", "timestamp after the data export", 0, s.created_future, "Set to null, flagged" + REBUILT),
    ("ticket_resolved", "timestamp after the data export", 0, s.resolved_future, "Set to null, flagged" + REBUILT),
    ("ticket_created", "blank value", s.created_blank_rebuilt, 0,
     "Rebuilt as resolved - resolution_time_hours where possible; rest stays null"),
    ("ticket_resolved", "blank value", s.resolved_blank_rebuilt, 0,
     "Rebuilt as created + resolution_time_hours where possible; rest stays null"),
    ("ticket_resolved", "resolved before created", 0, 0,
     f"Caused by future timestamps; after cleaning {s.resolved_before_created_left} remain"),
    ("resolution_time_hours", "resolution_time_hours does not match", 0, 0,
     "Caused by future timestamps; hours are consistent with every valid pair, so hours were kept"),
]
fix_rows = []
for column, start, fixed, flagged_n, action in plan:
    hit = m6_issues[(m6_issues["column"] == column) & m6_issues["issue"].str.startswith(start)]
    assert len(hit) == 1, (column, start)
    row = hit.iloc[0]
    fix_rows.append({"issue_id": row["issue_id"], "column": column, "issue": row["issue"],
                     "rows_in_m1": int(row["rows_affected"]), "rows_fixed": int(fixed),
                     "rows_flagged": int(flagged_n), "action_taken": action})
fix_log = pd.DataFrame(fix_rows)
assert set(fix_log["issue_id"]) == set(m6_issues["issue_id"]), "every M6 issue must be handled"
print(f"timestamps rebuilt from resolution_time_hours: created {s.created_derived:,}, resolved {s.resolved_derived:,}")
fix_log.drop(columns="issue")

timestamps rebuilt from resolution_time_hours: created 6,292, resolved 6,272


,issue_id,column,rows_in_m1,rows_fixed,rows_flagged,action_taken
0,ISS-070,sentiment,9000,6794,0,"Mapped with CASE/COALESCE: exact, leetspeak 3->e, unique prefix (neu/neg/pos)"
1,ISS-071,issue_type,8739,6494,0,"Mapped with CASE/COALESCE: exact, REVERSE() for reversed text, leetspeak, su..."
2,ISS-078,sentiment,2206,2206,0,TRIM
3,ISS-077,issue_type,2245,2245,0,TRIM
4,ISS-075,support_agent,2574,2574,0,TRIM
5,ISS-076,support_agent,2572,2572,0,"REGEXP_REPLACE removed symbols ('--', '@@')"
6,ISS-084,support_agent,1170,1032,139,Fixed only where the 'a' spelling exists as another agent name; others kept ...
7,ISS-072,ticket_created,5390,0,5390,"Not cast (pg_input_is_valid false): null, flagged; rebuilt from resolution_t..."
8,ISS-073,ticket_resolved,5324,0,5324,"Not cast (pg_input_is_valid false): null, flagged; rebuilt from resolution_t..."
9,ISS-081,ticket_created,1821,0,1821,"Set to null, flagged; rebuilt from resolution_time_hours when the other time..."


## 7. Export and validate

In [10]:
tickets = query("SELECT * FROM clean.tickets ORDER BY ticket_id")
tickets["resolution_time_hours"] = tickets["resolution_time_hours"].astype(float)
tickets["dq_flags"] = tickets["dq_flags"].fillna("")
flags = query("SELECT * FROM clean.ticket_flags ORDER BY flag_type, reason, ticket_id")
flags = flags.rename(columns={"column_name": "column"})  # same header as the other flagged sheets

counts = query("""
SELECT 'customers' AS t, COUNT(*) AS n FROM clean.customers UNION ALL
SELECT 'customer_devices', COUNT(*) FROM clean.customer_devices UNION ALL
SELECT 'products', COUNT(*) FROM clean.products UNION ALL
SELECT 'orders', COUNT(*) FROM clean.orders UNION ALL
SELECT 'clickstream', COUNT(*) FROM clean.clickstream UNION ALL
SELECT 'tickets', COUNT(*) FROM clean.tickets
""").set_index("t")["n"]

checks = {
    "30,000 tickets, ticket_id unique": len(tickets) == 30_000 and tickets["ticket_id"].is_unique,
    "issue_type standard": tickets["issue_type"].isin(["delay", "payment", "product", "refund"]).all(),
    "sentiment standard": tickets["sentiment"].isin(["negative", "neutral", "positive"]).all(),
    "timestamps not after the export day": (tickets[["ticket_created", "ticket_resolved"]].max().max()
                                           < pd.Timestamp("2025-12-03")),
    "no ticket resolved before created": not (tickets["ticket_resolved"] < tickets["ticket_created"]).any(),
    "agent names clean (letters, spaces, . ' -)": tickets["support_agent"].str.fullmatch(r"[A-Za-z][A-Za-z .'-]*[A-Za-z.]").all(),
    "all primary and foreign key checks pass":
        integrity.loc[integrity["check_type"].isin(["primary key", "foreign key"]), "status"].eq("pass").all(),
    "database holds every clean row": counts.to_dict() == {
        "customers": 48_200, "customer_devices": 51_854, "products": 500,
        "orders": 300_000, "clickstream": 500_000, "tickets": 30_000},
}
validation = pd.DataFrame({"check": checks.keys(), "passed": checks.values()})
assert validation["passed"].all(), validation
print(counts.to_string())
validation

t
products               500
tickets              30000
customer_devices     51854
customers            48200
orders              300000
clickstream         500000


,check,passed
0,"30,000 tickets, ticket_id unique",True
1,issue_type standard,True
2,sentiment standard,True
3,timestamps not after the export day,True
4,no ticket resolved before created,True
5,"agent names clean (letters, spaces, . ' -)",True
6,all primary and foreign key checks pass,True
7,database holds every clean row,True


In [11]:
def write_xlsx(path, sheets):
    with pd.ExcelWriter(path, engine="xlsxwriter", datetime_format="yyyy-mm-dd hh:mm:ss") as writer:
        for name, data in sheets.items():
            data.to_excel(writer, sheet_name=name, index=False)
            ws = writer.sheets[name]
            ws.freeze_panes(1, 0)
            ws.autofilter(0, 0, len(data), len(data.columns) - 1)
            for i, col in enumerate(data.columns):
                width = data[col].head(2000).astype(str).str.len().max() if len(data) else 0
                ws.set_column(i, i, min(max(width, len(col)) + 2, 70))
    print("saved:", path.resolve())


write_xlsx(CLEAN_DIR / "tickets_clean.xlsx", {"tickets": tickets, "flagged": flags})
write_xlsx(OUT_DIR / "m6_fix_log.xlsx", {"fix_log": fix_log, "integrity_checks": integrity, "validation": validation})

CSV_DIR = CLEAN_DIR / "csv"
CSV_DIR.mkdir(exist_ok=True)
tickets.to_csv(CSV_DIR / "tickets_clean.csv", index=False, encoding="utf-8", date_format="%Y-%m-%d %H:%M:%S")
print("saved CSV: tickets_clean.csv")

back = pd.read_excel(CLEAN_DIR / "tickets_clean.xlsx", sheet_name="tickets", dtype=str)
expected = tickets.replace("", None).notna().sum()  # empty strings are stored as empty cells
assert len(back) == len(tickets) and back.notna().sum().to_dict() == expected.to_dict()
print(f"read-back OK: {len(back):,} rows, filled cells per column identical to memory")

saved: C:\Users\msibr\OneDrive\Upwork\customer360-data-cleaning\03_clean_data_and_report\clean_data\tickets_clean.xlsx
saved: C:\Users\msibr\OneDrive\Upwork\customer360-data-cleaning\02_process\06_cleaning_tickets-sql\output\m6_fix_log.xlsx
saved CSV: tickets_clean.csv


read-back OK: 30,000 rows, filled cells per column identical to memory


## 8. Export the clean database

The `clean` schema (all 6 clean tables, the ticket flags and the integrity check results) is exported with `pg_dump` in PostgreSQL's compressed custom format. A client restores it into their own database with:

```
pg_restore --no-owner -d <their_database> customer360_clean.dump
```

In [12]:
PG_BIN = Path(r"C:\Program Files\PostgreSQL\18\bin")
pg_dump = shutil.which("pg_dump") or str(PG_BIN / "pg_dump.exe")
pg_restore = shutil.which("pg_restore") or str(PG_BIN / "pg_restore.exe")

DB_DIR = CLEAN_DIR / "database"
DB_DIR.mkdir(exist_ok=True)
dump = DB_DIR / "customer360_clean.dump"
subprocess.run([pg_dump, "-h", "localhost", "-p", "5432", "-U", "postgres", "-d", DB,
                "--schema=clean", "--format=custom", "--no-owner", "--no-privileges", "-f", str(dump)], check=True)

listing = subprocess.run([pg_restore, "--list", str(dump)], capture_output=True, text=True, check=True).stdout
tables = sorted({line.split(" TABLE clean ")[1].split()[0]
                 for line in listing.splitlines() if " TABLE clean " in line and "TABLE DATA" not in line})
print(f"saved: {dump.resolve()} ({dump.stat().st_size / 1024**2:.1f} MB)")
print("tables in dump:", tables)
assert {"customers", "customer_devices", "products", "orders", "clickstream", "tickets"} <= set(tables)
conn.close()

saved: C:\Users\msibr\OneDrive\Upwork\customer360-data-cleaning\03_clean_data_and_report\clean_data\database\customer360_clean.dump (71.3 MB)
tables in dump: ['clickstream', 'customer_devices', 'customers', 'integrity_checks', 'orders', 'products', 'ticket_flags', 'tickets']
